# Proyecto final - Parte opcional
## Análisis de satisfacción de app con apoyo de LLM

**Alumno:** Gerardo Hernández Chávez  
**Programa:** Máster en Data Science y Big Data  
**Módulo:** Sistemas de Recomendación Inteligente

## 1. Objetivo

El notebook analiza datos semanales de uso y satisfacción de una app. A partir de indicadores verificables construye un prompt ejecutivo y genera un informe de 6 a 10 líneas para un director no técnico. El flujo admite Qwen 2.5 0.5B Instruct, utilizado en los ejemplos previos, y mantiene una plantilla local cuando el modelo no está disponible.

## 2. Instalación opcional del LLM en Colab

El análisis numérico no necesita estas dependencias. Solo deben instalarse si se desea ejecutar Qwen localmente en Colab.

In [1]:
# Instalación opcional para usar el LLM real en Google Colab.
# !pip install transformers torch accelerate sentencepiece

## 3. Importación de librerías

In [2]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.precision", 3)

## 4. Carga y preparación del dataset

In [3]:
RUTA_CLASE_4 = Path(
    "/Users/gerardo/Desktop/mi_proyecto/Cursos/Master IEBS - Ciencia de Datos/"
    "14. Sistemas de Recomendación Inteligente/Clase 4"
)
candidatos = [
    Path("/content/satisfaccion_app.csv"),
    Path("satisfaccion_app.csv"),
    RUTA_CLASE_4 / "satisfaccion_app.csv",
]
satisfaccion_path = next((ruta for ruta in candidatos if ruta.exists()), None)

if satisfaccion_path is None:
    raise FileNotFoundError(
        "No se encontró satisfaccion_app.csv. En Colab, carga el archivo en /content."
    )

satisfaccion = pd.read_csv(satisfaccion_path)
satisfaccion.columns = [
    columna.strip().lower().replace(" ", "_")
    for columna in satisfaccion.columns
]
satisfaccion["fecha"] = pd.to_datetime(satisfaccion["fecha"], errors="raise")
satisfaccion["satisfaccion"] = pd.to_numeric(
    satisfaccion["satisfaccion"].astype(str).str.replace("%", "", regex=False),
    errors="raise",
)

display(satisfaccion.head())
print(f"Dimensiones: {satisfaccion.shape}")

,fecha,usuarios_activos,nuevos_usuarios,satisfaccion
0,2025-01-01,1200,300,78
1,2025-01-08,1320,280,80
2,2025-01-15,1400,350,82
3,2025-01-22,1500,370,83
4,2025-01-29,1580,360,85


Dimensiones: (6, 4)


In [4]:
calidad = pd.DataFrame(
    {
        "tipo": satisfaccion.dtypes.astype(str),
        "nulos": satisfaccion.isna().sum(),
    }
)
display(calidad)

,tipo,nulos
fecha,datetime64[us],0
usuarios_activos,int64,0
nuevos_usuarios,int64,0
satisfaccion,int64,0


## 5. Cálculo de indicadores

In [5]:
satisfaccion["variacion_usuarios_activos_pct"] = (
    satisfaccion["usuarios_activos"].pct_change() * 100
)
satisfaccion["variacion_nuevos_usuarios_pct"] = (
    satisfaccion["nuevos_usuarios"].pct_change() * 100
)

mejor_semana = satisfaccion.loc[satisfaccion["satisfaccion"].idxmax()]
cambio_satisfaccion_pp = (
    satisfaccion["satisfaccion"].iloc[-1]
    - satisfaccion["satisfaccion"].iloc[0]
)

resumen_indicadores = {
    "periodo_inicio": satisfaccion["fecha"].min().strftime("%Y-%m-%d"),
    "periodo_fin": satisfaccion["fecha"].max().strftime("%Y-%m-%d"),
    "promedio_usuarios_activos": float(satisfaccion["usuarios_activos"].mean()),
    "variacion_promedio_usuarios_activos_pct": float(
        satisfaccion["variacion_usuarios_activos_pct"].mean()
    ),
    "variacion_ultima_semana_usuarios_activos_pct": float(
        satisfaccion["variacion_usuarios_activos_pct"].iloc[-1]
    ),
    "promedio_tasa_satisfaccion": float(satisfaccion["satisfaccion"].mean()),
    "semana_mejor_satisfaccion": mejor_semana["fecha"].strftime("%Y-%m-%d"),
    "mejor_tasa_satisfaccion": float(mejor_semana["satisfaccion"]),
    "crecimiento_promedio_nuevos_usuarios_pct": float(
        satisfaccion["variacion_nuevos_usuarios_pct"].mean()
    ),
    "cambio_satisfaccion_periodo_pp": float(cambio_satisfaccion_pp),
}

display(satisfaccion)
display(pd.Series(resumen_indicadores, name="valor").to_frame())

,fecha,usuarios_activos,nuevos_usuarios,satisfaccion,variacion_usuarios_activos_pct,variacion_nuevos_usuarios_pct
0,2025-01-01,1200,300,78,NaN,NaN
1,2025-01-08,1320,280,80,10.000,-6.667
2,2025-01-15,1400,350,82,6.061,25.000
3,2025-01-22,1500,370,83,7.143,5.714
4,2025-01-29,1580,360,85,5.333,-2.703
5,2025-02-05,1620,340,84,2.532,-5.556


,valor
periodo_inicio,2025-01-01
periodo_fin,2025-02-05
promedio_usuarios_activos,1436.667
variacion_promedio_usuarios_activos_pct,6.214
variacion_ultima_semana_usuarios_activos_pct,2.532
promedio_tasa_satisfaccion,82.0
semana_mejor_satisfaccion,2025-01-29
mejor_tasa_satisfaccion,85.0
crecimiento_promedio_nuevos_usuarios_pct,3.158
cambio_satisfaccion_periodo_pp,6.0


## 6. Mini prompt para el informe ejecutivo

El prompt conserva el patrón de los ejemplos de Qwen: define un rol, entrega datos concretos, limita la extensión y especifica el formato esperado. También prohíbe inventar causas; el modelo puede interpretar tendencias, pero no atribuirlas a eventos que el dataset no contiene.

In [6]:
def crear_prompt_ejecutivo(resumen):
    "Construye un prompt acotado y verificable para el informe."
    return f'''
Actúa como asistente de análisis de negocio para una aplicación digital.

Datos calculados en Python:
- Periodo: {resumen['periodo_inicio']} a {resumen['periodo_fin']}.
- Usuarios activos promedio: {resumen['promedio_usuarios_activos']:.1f}.
- Variación semanal promedio de usuarios activos: {resumen['variacion_promedio_usuarios_activos_pct']:.2f}%.
- Variación de usuarios activos en la última semana: {resumen['variacion_ultima_semana_usuarios_activos_pct']:.2f}%.
- Satisfacción promedio: {resumen['promedio_tasa_satisfaccion']:.1f}%.
- Mejor semana: {resumen['semana_mejor_satisfaccion']} con {resumen['mejor_tasa_satisfaccion']:.1f}%.
- Crecimiento semanal promedio de nuevos usuarios: {resumen['crecimiento_promedio_nuevos_usuarios_pct']:.2f}%.
- Cambio de satisfacción entre el inicio y el final: {resumen['cambio_satisfaccion_periodo_pp']:.1f} puntos porcentuales.

Redacta un informe ejecutivo de 6 a 10 líneas para un director no técnico.
Usa lenguaje claro, interpreta la tendencia y termina con una recomendación de negocio.
No inventes causas, campañas ni hechos que no aparecen en los datos.
Devuelve solo el informe, sin título ni viñetas.
'''.strip()

prompt_ejecutivo = crear_prompt_ejecutivo(resumen_indicadores)
print(prompt_ejecutivo)

Actúa como asistente de análisis de negocio para una aplicación digital.

Datos calculados en Python:
- Periodo: 2025-01-01 a 2025-02-05.
- Usuarios activos promedio: 1436.7.
- Variación semanal promedio de usuarios activos: 6.21%.
- Variación de usuarios activos en la última semana: 2.53%.
- Satisfacción promedio: 82.0%.
- Mejor semana: 2025-01-29 con 85.0%.
- Crecimiento semanal promedio de nuevos usuarios: 3.16%.
- Cambio de satisfacción entre el inicio y el final: 6.0 puntos porcentuales.

Redacta un informe ejecutivo de 6 a 10 líneas para un director no técnico.
Usa lenguaje claro, interpreta la tendencia y termina con una recomendación de negocio.
No inventes causas, campañas ni hechos que no aparecen en los datos.
Devuelve solo el informe, sin título ni viñetas.


## 7. Generación con Qwen o plantilla local

In [7]:
MODELO_LLM = "Qwen/Qwen2.5-0.5B-Instruct"
USAR_LLM_REAL = os.getenv("USAR_LLM_REAL", "0") == "1"

def ejecutar_qwen(prompt, max_tokens=260):
    "Carga Qwen bajo demanda y genera una respuesta en CPU."
    from transformers import AutoModelForCausalLM, AutoTokenizer
    import torch

    tokenizer = AutoTokenizer.from_pretrained(MODELO_LLM)
    model = AutoModelForCausalLM.from_pretrained(
        MODELO_LLM,
        torch_dtype=torch.float32,
    )
    mensajes = [
        {"role": "system", "content": "Respondes en español con precisión y claridad."},
        {"role": "user", "content": prompt},
    ]
    texto = tokenizer.apply_chat_template(
        mensajes,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(texto, return_tensors="pt")
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_tokens,
        do_sample=False,
    )
    tokens_nuevos = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(tokens_nuevos, skip_special_tokens=True).strip()

def generar_informe_local(resumen):
    "Genera ocho líneas basadas únicamente en los indicadores calculados."
    tendencia = "aumentó" if resumen["cambio_satisfaccion_periodo_pp"] >= 0 else "disminuyó"
    return "\n".join(
        [
            f"El periodo analizado comprende del {resumen['periodo_inicio']} al {resumen['periodo_fin']}.",
            f"La aplicación registró un promedio de {resumen['promedio_usuarios_activos']:.0f} usuarios activos por semana.",
            f"Los usuarios activos crecieron en promedio {resumen['variacion_promedio_usuarios_activos_pct']:.2f}% semanal.",
            f"En la última semana el avance fue de {resumen['variacion_ultima_semana_usuarios_activos_pct']:.2f}%, señal de un crecimiento más moderado.",
            f"La satisfacción promedio se ubicó en {resumen['promedio_tasa_satisfaccion']:.1f}%.",
            f"El mejor resultado ocurrió el {resumen['semana_mejor_satisfaccion']}, con {resumen['mejor_tasa_satisfaccion']:.1f}% de satisfacción.",
            f"Durante todo el periodo, la satisfacción {tendencia} {abs(resumen['cambio_satisfaccion_periodo_pp']):.1f} puntos porcentuales.",
            "Se recomienda investigar los factores de la mejor semana y vigilar la desaceleración de usuarios activos antes de definir nuevas acciones.",
        ]
    )

def generar_informe_llm(resumen, usar_llm=USAR_LLM_REAL):
    "Usa Qwen si está habilitado; ante cualquier fallo conserva una salida reproducible."
    prompt = crear_prompt_ejecutivo(resumen)
    if usar_llm:
        try:
            return ejecutar_qwen(prompt), f"LLM real: {MODELO_LLM}"
        except Exception as error:
            print(f"No fue posible ejecutar Qwen: {error}")
            print("Se aplicará la plantilla local.")
    return generar_informe_local(resumen), "Fallback local basado en plantilla"

## 8. Informe ejecutivo

In [8]:
informe_ejecutivo, fuente_informe = generar_informe_llm(resumen_indicadores)

print(f"Fuente: {fuente_informe}\n")
print(informe_ejecutivo)

Fuente: Fallback local basado en plantilla

El periodo analizado comprende del 2025-01-01 al 2025-02-05.
La aplicación registró un promedio de 1437 usuarios activos por semana.
Los usuarios activos crecieron en promedio 6.21% semanal.
En la última semana el avance fue de 2.53%, señal de un crecimiento más moderado.
La satisfacción promedio se ubicó en 82.0%.
El mejor resultado ocurrió el 2025-01-29, con 85.0% de satisfacción.
Durante todo el periodo, la satisfacción aumentó 6.0 puntos porcentuales.
Se recomienda investigar los factores de la mejor semana y vigilar la desaceleración de usuarios activos antes de definir nuevas acciones.


## 9. Mini prompt para revisar la narrativa

Una segunda instrucción puede pedir al modelo que contraste el texto con los indicadores. No reemplaza la revisión humana, pero ayuda a fijar criterios de coherencia, fidelidad y utilidad.

In [9]:
def crear_prompt_revision(resumen, informe):
    return f'''
Revisa el siguiente informe ejecutivo contra los indicadores proporcionados.

Indicadores: {resumen}

Informe:
{informe}

Responde en tres apartados breves:
1. Coherencia general.
2. Afirmaciones respaldadas o no respaldadas por los datos.
3. Una mejora concreta.
No recalcules ni inventes información externa.
'''.strip()

prompt_revision = crear_prompt_revision(resumen_indicadores, informe_ejecutivo)
print(prompt_revision)

Revisa el siguiente informe ejecutivo contra los indicadores proporcionados.

Indicadores: {'periodo_inicio': '2025-01-01', 'periodo_fin': '2025-02-05', 'promedio_usuarios_activos': 1436.6666666666667, 'variacion_promedio_usuarios_activos_pct': 6.213688421283353, 'variacion_ultima_semana_usuarios_activos_pct': 2.5316455696202445, 'promedio_tasa_satisfaccion': 82.0, 'semana_mejor_satisfaccion': '2025-01-29', 'mejor_tasa_satisfaccion': 85.0, 'crecimiento_promedio_nuevos_usuarios_pct': 3.157872157872159, 'cambio_satisfaccion_periodo_pp': 6.0}

Informe:
El periodo analizado comprende del 2025-01-01 al 2025-02-05.
La aplicación registró un promedio de 1437 usuarios activos por semana.
Los usuarios activos crecieron en promedio 6.21% semanal.
En la última semana el avance fue de 2.53%, señal de un crecimiento más moderado.
La satisfacción promedio se ubicó en 82.0%.
El mejor resultado ocurrió el 2025-01-29, con 85.0% de satisfacción.
Durante todo el periodo, la satisfacción aumentó 6.0 punto

## 10. Evaluación de la narrativa

In [10]:
evaluacion_narrativa = pd.DataFrame(
    {
        "pregunta": [
            "¿El informe es coherente?",
            "¿Interpreta correctamente los datos?",
            "¿Qué mejorarías o agregarías?",
        ],
        "respuesta": [
            "Sí. Sigue una secuencia lógica: volumen de usuarios, evolución, satisfacción y recomendación.",
            "Sí. Las cifras y tendencias se derivan del diccionario calculado; no atribuye causas no observadas.",
            "Añadiría objetivos de negocio, comparación con periodos anteriores y segmentación de usuarios antes de recomendar acciones definitivas.",
        ],
    }
)

display(evaluacion_narrativa)

,pregunta,respuesta
0,¿El informe es coherente?,Sí. Sigue una secuencia lógica: volumen de usu...
1,¿Interpreta correctamente los datos?,Sí. Las cifras y tendencias se derivan del dic...
2,¿Qué mejorarías o agregarías?,"Añadiría objetivos de negocio, comparación con..."


## 11. Conclusión

Se cargó y validó el dataset semanal, se calcularon indicadores de actividad, captación y satisfacción, y se produjo una narrativa ejecutiva de ocho líneas. El prompt sigue el enfoque de Qwen utilizado en ejemplos anteriores y restringe la respuesta a datos comprobables. Cuando el modelo no está disponible, la plantilla local mantiene la ejecución completa y deja identificada la fuente del texto. Un LLM puede traducir métricas a lenguaje de negocio, pero la narrativa siempre debe contrastarse con los cálculos originales y con el contexto que el dataset no contiene.